In [1]:
from pyspark.sql import functions as F
from pyspark.sql import Window

reg01 = spark.sql("SELECT * FROM lh_cidade_inteligente_osasco.bronze_cad_unico_reg01")
reg04 = spark.sql("SELECT * FROM lh_cidade_inteligente_osasco.bronze_cad_unico_reg04")


def mapear_codigo_descricao(df, coluna_codigo, coluna_descricao, mapeamento, cast_type="int"):
    """
    Cria coluna de descrição a partir de um dicionário código->descrição
    - Faz cast do código
    - Faz o map com create_map + element_at
    """
    # cria um literal MapType: {1:"...", 2:"..."}
    mapping_expr = F.create_map(*[x for kv in mapeamento.items() for x in (F.lit(kv[0]), F.lit(kv[1]))])

    return (
        df
        .withColumn(coluna_codigo, F.col(coluna_codigo).cast(cast_type))
        .withColumn(coluna_descricao, F.element_at(mapping_expr, F.col(coluna_codigo)))
    )


mapeamento_estado_cadastral_fam = {
    1: "Em cadastramento",
    2: "Sem registro civil",
    3: "Cadastrado",
    4: "Excluído",
}
mapeamento_estado_cadastral_memb = {
    1: "Em cadastramento",
    2: "Sem registro civil",
    3: "Cadastrado",
    4: "Excluído",
    5: "Aguardando atribuição NIS",
    6: "Aguardando alteração de caracterização",
    7: "Aguardando CPF",
}
reg01_ = mapear_codigo_descricao(
    reg01,
    "cod_est_cadastral_fam",
    "desc_cod_est_cadastral_fam",
    mapeamento_estado_cadastral_fam,
)
reg04_ = mapear_codigo_descricao(
    reg04,
    "cod_est_cadastral_memb",
    "desc_cod_est_cadastral_memb",
    mapeamento_estado_cadastral_memb,
)

StatementMeta(, 3b283e24-33e7-4381-96b0-f87ac3ce04e8, 3, Finished, Available, Finished, False)

In [2]:
cod_parentesco_rf_pessoa = {
    "01": "Pessoa responsável pela Unidade Familiar - RF",
    "02": "Cônjuge ou companheiro(a)",
    "03": "Filho(a)",
    "04": "Enteado(a)",
    "05": "Neto(a) ou bisneto(a)",
    "06": "Pai ou mãe",
    "07": "Sogro(a)",
    "08": "Irmão ou irmã",
    "09": "Genro ou nora",
    "10": "Outro parente",
    "11": "Não parente",
}

# transforma dict em expressão Spark Map
mapping_expr = F.create_map(
    *[F.lit(x) for x in sum(cod_parentesco_rf_pessoa.items(), ())]
)

reg04_ = reg04_.withColumn(
    "desc_cod_parentesco_rf_pessoa",
    mapping_expr[F.col("cod_parentesco_rf_pessoa")]
)

StatementMeta(, 3b283e24-33e7-4381-96b0-f87ac3ce04e8, 4, Finished, Available, Finished, False)

In [3]:
def construir_transicoes_spark(
    df,
    keys,
    col_mes,
    col_status,
    prefixo="",
    considerar_entrada_informal=True,
):
    out = df

    # 0) Normaliza tipo de data (se col_mes for timestamp/string)
    out = out.withColumn(col_mes, F.to_date(F.col(col_mes)))

    # (opcional mas recomendado) garante 1 linha por (keys, mes)
    out = out.dropDuplicates(keys + [col_mes])

    # 1) LAG por chave (snapshot anterior disponível da chave)
    w_key = Window.partitionBy(*keys).orderBy(F.col(col_mes))

    out = (
        out.withColumn(f"{prefixo}status_prev", F.lag(F.col(col_status), 1).over(w_key))
           .withColumn(f"{prefixo}mes_prev",    F.lag(F.col(col_mes), 1).over(w_key))
           .withColumn(
               f"{prefixo}gap_dias",
               F.when(F.col(f"{prefixo}mes_prev").isNotNull(),
                      F.datediff(F.col(col_mes), F.col(f"{prefixo}mes_prev")))
           )
           .withColumn(
               f"{prefixo}mudou_status",
               (F.col(f"{prefixo}status_prev").isNotNull()) &
               (F.col(col_status) != F.col(f"{prefixo}status_prev"))
           )
    )

    # 2) Próximo/anterior mês GLOBAL disponível no dataset
    meses = out.select(F.col(col_mes).alias("_mes")).where(F.col("_mes").isNotNull()).distinct()
    w_mes = Window.orderBy(F.col("_mes"))

    meses_map = (
        meses.withColumn("_mes_next_global", F.lead(F.col("_mes"), 1).over(w_mes))
             .withColumn("_mes_prev_global", F.lag (F.col("_mes"), 1).over(w_mes))
    )

    out = (
        out.join(meses_map, out[col_mes] == meses_map["_mes"], "left")
           .drop("_mes")
           .withColumnRenamed("_mes_next_global", f"{prefixo}mes_next_global")
           .withColumnRenamed("_mes_prev_global", f"{prefixo}mes_prev_global")
    )

    # 3) Tabela de existência: (keys, mes) distintos
    base_exist = out.select(*keys, F.col(col_mes).alias("_mes_exist")).distinct()

    # ---------- presente_no_next / saida_informal ----------
    alvo_next = (
        out.select(*keys, F.col(col_mes).alias("_mes_atual"),
                   F.col(f"{prefixo}mes_next_global").alias("_mes_target"))
           .where(F.col("_mes_target").isNotNull())
    )

    # renomeia base para join só por nomes
    base_next = base_exist.withColumnRenamed("_mes_exist", "_mes_target")

    presenca_next = (
        alvo_next.join(base_next, on=keys + ["_mes_target"], how="left")
                 .withColumn(f"{prefixo}presente_no_next", F.col("_mes_target").isNotNull() & F.col("_mes_target").isNotNull())  # dummy
    )

    # a linha acima não serve: precisamos de um marcador vindo do lado direito
    # solução: criar coluna flag no base_next
    base_next = base_next.withColumn("_existe", F.lit(1))

    presenca_next = (
        alvo_next.join(base_next, on=keys + ["_mes_target"], how="left")
                 .withColumn(f"{prefixo}presente_no_next", F.col("_existe").isNotNull())
                 .select(*keys, F.col("_mes_atual").alias(col_mes), F.col(f"{prefixo}presente_no_next"))
    )

    out = (
        out.join(presenca_next, on=keys + [col_mes], how="left")
           .withColumn(f"{prefixo}presente_no_next", F.coalesce(F.col(f"{prefixo}presente_no_next"), F.lit(False)))
           .withColumn(
               f"{prefixo}saida_informal",
               (F.col(f"{prefixo}mes_next_global").isNotNull()) & (~F.col(f"{prefixo}presente_no_next"))
           )
    )

    # ---------- presente_no_prev_global / entrada_informal ----------
    if considerar_entrada_informal:
        alvo_prev = (
            out.select(*keys, F.col(col_mes).alias("_mes_atual"),
                       F.col(f"{prefixo}mes_prev_global").alias("_mes_target_prev"))
               .where(F.col("_mes_target_prev").isNotNull())
        )

        base_prev = base_exist.withColumnRenamed("_mes_exist", "_mes_target_prev").withColumn("_existe_prev", F.lit(1))

        presenca_prev = (
            alvo_prev.join(base_prev, on=keys + ["_mes_target_prev"], how="left")
                     .withColumn(f"{prefixo}presente_no_prev_global", F.col("_existe_prev").isNotNull())
                     .select(*keys, F.col("_mes_atual").alias(col_mes), F.col(f"{prefixo}presente_no_prev_global"))
        )

        out = (
            out.join(presenca_prev, on=keys + [col_mes], how="left")
               .withColumn(f"{prefixo}presente_no_prev_global",
                           F.coalesce(F.col(f"{prefixo}presente_no_prev_global"), F.lit(False)))
               .withColumn(
                   f"{prefixo}entrada_informal",
                   (F.col(f"{prefixo}mes_prev_global").isNotNull()) & (~F.col(f"{prefixo}presente_no_prev_global"))
               )
        )

    # 4) Tipo de transição (prioriza saída informal)
    out = out.withColumn(
        f"{prefixo}tipo_transicao",
        F.when(F.col(f"{prefixo}saida_informal"), F.lit("Saída informal"))
         .when(F.col(f"{prefixo}status_prev").isNull(), F.lit("Primeira aparição"))
         .when((F.col(f"{prefixo}status_prev") == 3) & (F.col(col_status) == 4), F.lit("Exclusão (3→4)"))
         .when((F.col(f"{prefixo}status_prev") == 4) & (F.col(col_status) == 3), F.lit("Reativação (4→3)"))
         .when(F.col(col_status) != F.col(f"{prefixo}status_prev"),
               F.concat(F.lit("Outra mudança ("), F.col(f"{prefixo}status_prev").cast("int"),
                        F.lit("→"), F.col(col_status).cast("int"), F.lit(")")))
         .otherwise(F.lit("Sem mudança"))
    )

    return out

StatementMeta(, 3b283e24-33e7-4381-96b0-f87ac3ce04e8, 5, Finished, Available, Finished, False)

In [4]:
fam_tr = construir_transicoes_spark(
    df=reg01_,
    keys=["cod_familiar_fam"],
    col_mes="versao_arquivo",
    col_status="cod_est_cadastral_fam",
    prefixo="fam_",
    considerar_entrada_informal=True
)
fam_tr = fam_tr.withColumn(
    "chave_familia_versao",
    F.concat(
        F.date_format(F.col("versao_arquivo"), "yyyyMMdd"),
        F.lit("_"),                                     
        F.col("cod_familiar_fam")
    )
)

StatementMeta(, 3b283e24-33e7-4381-96b0-f87ac3ce04e8, 6, Finished, Available, Finished, False)

In [5]:
memb_tr = construir_transicoes_spark(
    df=reg04_,
    keys=["cod_familiar_fam", "chv_nat_pes_atual"],
    col_mes="versao_arquivo",
    col_status="cod_est_cadastral_memb",
    prefixo="memb_",
    considerar_entrada_informal=True
)
memb_tr = memb_tr.withColumn(
    "chave_familia_versao",
    F.concat(
        F.date_format(F.col("versao_arquivo"), "yyyyMMdd"),
        F.lit("_"),                                     
        F.col("cod_familiar_fam")
    )
)

StatementMeta(, 3b283e24-33e7-4381-96b0-f87ac3ce04e8, 7, Finished, Available, Finished, False)

In [6]:

fam_tr = fam_tr.select("cod_familiar_fam", "desc_cod_est_cadastral_fam",  "dat_cadastramento_fam", "dat_alteracao_fam", "fam_tipo_transicao", "chave_familia_versao", "versao_arquivo")

fam_tr = (
    fam_tr
    .withColumn("dat_cadastramento_fam", F.to_date(fam_tr["dat_cadastramento_fam"], "ddMMyyyy"))
    .withColumn("dat_alteracao_fam", F.to_date(fam_tr["dat_alteracao_fam"], "ddMMyyyy"))
)

StatementMeta(, 3b283e24-33e7-4381-96b0-f87ac3ce04e8, 8, Finished, Available, Finished, False)

In [7]:
memb_tr = memb_tr.select(
    "chv_nat_pes_atual", 
    "nom_pessoa", 
    "desc_cod_parentesco_rf_pessoa", 
    "desc_cod_est_cadastral_memb", 
    "dta_cadastramento_memb", 
    "dta_atual_memb", 
    "memb_tipo_transicao", 
    "chave_familia_versao",
    "versao_arquivo"
)

memb_tr = (
    memb_tr
    .withColumn("dta_cadastramento_memb", F.to_date(memb_tr["dta_cadastramento_memb"], "ddMMyyyy"))
    .withColumn("dta_atual_memb", F.to_date(memb_tr["dta_atual_memb"], "ddMMyyyy"))
)

StatementMeta(, 3b283e24-33e7-4381-96b0-f87ac3ce04e8, 9, Finished, Available, Finished, False)

In [8]:
tabelas_silver = {
    "silver_cad_unico_reg01": fam_tr,
    "silver_cad_unico_reg04": memb_tr,
}

for nome_tabela, df in tabelas_silver.items():
    (
        df.write
        .mode("overwrite")
        .format("delta")
        .option("overwriteSchema", "true")
        .saveAsTable(nome_tabela)
    )

StatementMeta(, 3b283e24-33e7-4381-96b0-f87ac3ce04e8, 10, Submitted, Running, Running, True)